Priprema EgoHands dataset za YOLO segmentaciju: ucitava `metadata.mat`, pravi train/val/test split, konvertuje poligone u YOLO format i generise `data.yaml`. 

In [1]:
import os
import random
import shutil
from pathlib import Path

#import cv2
import numpy as np
from scipy.io import loadmat

# ============================================================
# CONFIG
# ============================================================

# Edit this to your extracted EgoHands folder (must contain metadata.mat + _LABELLED_SAMPLES/)
DATASET_ROOT = Path(r"data\EgoHands")

NOTEBOOK_DIR = Path.cwd().resolve()
LABELLED_SAMPLES = DATASET_ROOT / "_LABELLED_SAMPLES"
METADATA_PATH = DATASET_ROOT / "metadata.mat"
print(METADATA_PATH)
OUTPUT_DATASET = NOTEBOOK_DIR / "data/egohands_yolo"

# split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

# jedna klasa
CLASS_ID = 0
HAND_KEYS = ("myleft", "myright", "yourleft", "yourright")

# ============================================================
# CREATE OUTPUT FOLDERS
# ============================================================

for split in ["train", "val", "test"]:
    (OUTPUT_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

# ============================================================
# LOAD MATLAB METADATA
# ============================================================

print("Loading metadata...")

if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"EgoHands metadata not found at {METADATA_PATH.resolve()}\n"
        f"Edit DATASET_ROOT in CONFIG to your extracted EgoHands folder."
    )

mat = loadmat(str(METADATA_PATH), simplify_cells=True)
videos = mat["video"]

print(f"Loaded {len(videos)} videos")

# ============================================================
# SPLIT BY VIDEOS
# ============================================================

random.seed(RANDOM_SEED)

video_indices = list(range(len(videos)))
random.shuffle(video_indices)

num_videos = len(video_indices)

train_end = int(num_videos * TRAIN_RATIO)
val_end = train_end + int(num_videos * VAL_RATIO)

train_indices = set(video_indices[:train_end])
val_indices = set(video_indices[train_end:val_end])
test_indices = set(video_indices[val_end:])

print("\n================================================")
print("DATASET SPLIT")
print("================================================")
print(f"Train videos: {len(train_indices)}")
print(f"Val videos:   {len(val_indices)}")
print(f"Test videos:  {len(test_indices)}")

# ============================================================
# HELPERS
# ============================================================

IMAGE_WIDTH = 1280
IMAGE_HEIGHT = 720


def normalize_polygon(poly_x, poly_y):
    points = []

    for x, y in zip(poly_x, poly_y):
        xn = float(x) / IMAGE_WIDTH
        yn = float(y) / IMAGE_HEIGHT

        xn = min(max(xn, 0.0), 1.0)
        yn = min(max(yn, 0.0), 1.0)

        points.extend([xn, yn])

    return points


# ============================================================
# PROCESS DATASET
# ============================================================

total_images = 0
total_instances = 0

for vid_idx, video in enumerate(videos):

    if vid_idx in train_indices:
        split = "train"
    elif vid_idx in val_indices:
        split = "val"
    else:
        split = "test"

    video_id = video["video_id"]

    print(f"\nProcessing video: {video_id}")

    video_folder = LABELLED_SAMPLES / video_id

    labelled_frames = video["labelled_frames"]

    for frame_data in labelled_frames:

        frame_id = frame_data["frame_num"]

        # image file name
        image_name = f"frame_{frame_id:04d}.jpg"

        image_src = video_folder / image_name

        # provera da li postoji slika
        if not image_src.is_file():
            # neka verzija dataset-a koristi image_XXXX.jpg
            image_name = f"image_{frame_id:04d}.jpg"
            image_src = video_folder / image_name

        if not image_src.is_file():
            print(f"Missing image: {image_src}")
            continue

        # novo ime slike
        unique_name = f"{video_id}_{frame_id:04d}"

        image_dst = OUTPUT_DATASET / "images" / split / f"{unique_name}.jpg"

        label_dst = OUTPUT_DATASET / "labels" / split / f"{unique_name}.txt"

        # copy image
        shutil.copy(str(image_src), str(image_dst))

        lines = []

        # ====================================================
        # EACH HAND INSTANCE
        # ====================================================

        for hand_key in HAND_KEYS:
            polygon = frame_data[hand_key]
            if polygon.size == 0 or len(polygon) < 3:
                continue

            try:
                x_points = polygon[:, 0]
                y_points = polygon[:, 1]

                coords = normalize_polygon(x_points, y_points)

                # YOLO segmentation format
                line = (
                    str(CLASS_ID) + " " +
                    " ".join([f"{p:.6f}" for p in coords])
                )

                lines.append(line)

                total_instances += 1

            except Exception as e:
                print("Polygon error:", e)
                continue

        # save label file
        with open(label_dst, "w") as f:
            for line in lines:
                f.write(line + "\n")

        total_images += 1

print("\n================================================")
print("DONE")
print("================================================")
print(f"Images: {total_images}")
print(f"Hand instances: {total_instances}")

# ============================================================
# CREATE data.yaml
# ============================================================

yaml_text = """
path: data/egohands_yolo

train: images/train
val: images/val
test: images/test

names:
  0: hand
"""

with open(OUTPUT_DATASET / "data.yaml", "w") as f:
    f.write(yaml_text)

print("\nCreated data.yaml")
print("\nDataset ready for YOLO segmentation training!")

data\EgoHands\metadata.mat
Loading metadata...
Loaded 48 videos

DATASET SPLIT
Train videos: 33
Val videos:   7
Test videos:  8

Processing video: CARDS_COURTYARD_B_T

Processing video: CARDS_COURTYARD_H_S

Processing video: CARDS_COURTYARD_S_H

Processing video: CARDS_COURTYARD_T_B

Processing video: CARDS_LIVINGROOM_B_T

Processing video: CARDS_LIVINGROOM_H_S

Processing video: CARDS_LIVINGROOM_S_H

Processing video: CARDS_LIVINGROOM_T_B

Processing video: CARDS_OFFICE_B_S

Processing video: CARDS_OFFICE_H_T

Processing video: CARDS_OFFICE_S_B

Processing video: CARDS_OFFICE_T_H

Processing video: CHESS_COURTYARD_B_T

Processing video: CHESS_COURTYARD_H_S

Processing video: CHESS_COURTYARD_S_H

Processing video: CHESS_COURTYARD_T_B

Processing video: CHESS_LIVINGROOM_B_S

Processing video: CHESS_LIVINGROOM_H_T

Processing video: CHESS_LIVINGROOM_S_B

Processing video: CHESS_LIVINGROOM_T_H

Processing video: CHESS_OFFICE_B_S

Processing video: CHESS_OFFICE_H_T

Processing video: CHESS